# MA-POCA Baseline — Cooperative Push Block (stock `poca`, 30M steps)

Trains the **pristine stock MA-POCA baseline** for the Comm-MAPOCA benchmark.
Mirror of `pushblock_comm_fullobs_s1`: same scene budget (5000 env steps), same
hyperparameters, same parallelism (14 in-scene arenas x `--num-envs=2` = 28 arenas),
only the trainer (`poca` vs `comm_mapoca`) and communication differ.

## Session checklist (set in the right panel BEFORE running)
1. **Add Input** -> Your Datasets -> `mapoca-baseline-data` (contains `PBBaseline_Linux.zip`, auto-extracted)
2. **Accelerator: None** (workload is CPU-bound; saves the 30h/week GPU quota)
3. **Internet: On** (needed for git clone + pip)
4. **Add-ons -> Secrets** -> attach `GITHUB_TOKEN` (fine-grained PAT, Contents: Read-only)

Run cells top to bottom. For a long unattended run: **Save Version -> Save & Run All (Commit)**.
Expect ~16h for 30M fresh steps at ~500 steps/s, so plan one `--resume` chain after the
background limit cuts the first commit (checkpoints save every 500k steps; nothing is lost).

In [ ]:
# Cell 1 - install uv, get exact Python 3.10.12 (ML-Agents R23 requirement), clone the repo
!pip -q install uv
!uv python install 3.10.12
!rm -rf repo
from kaggle_secrets import UserSecretsClient
import os
os.environ['GT'] = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone https://$GT@github.com/Lihaj/4th-Year-Research-COMM-MAPOCA.git repo

In [ ]:
# Cell 2 - venv + the proven-working stack (matches the local machine exactly)
# setuptools<81 restores pkg_resources; torch pinned to 2.5.1 (avoids the
# onnxscript export path of newer torch). CPU wheels are fine: Accelerator=None.
!uv venv --python 3.10.12 mlagents-venv
!uv pip install --python mlagents-venv -e repo/ml-agents-envs -e repo/ml-agents
!uv pip install --python mlagents-venv "setuptools<81"
!uv pip install --python mlagents-venv torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

In [ ]:
# Cell 2b - verify the dataset's real folder nesting BEFORE the copy step
!ls -R /kaggle/input/datasets/lihajwicky/mapoca-baseline-data | head -40

In [ ]:
# Cell 3 - copy the game build into the working dir (adjust path if Cell 2b differs)
DATA = "/kaggle/input/datasets/lihajwicky/mapoca-baseline-data"
!mkdir -p env results
!cp -r "{DATA}/PBBaseline_Linux/." env/
!chmod +x env/PBBaseline.x86_64

In [ ]:
# Cell 4 - verify the binary is a valid Linux executable and boots headless
!file env/PBBaseline.x86_64
!timeout 15 ./env/PBBaseline.x86_64 -batchmode -nographics -logFile - || echo "exit code: $?"
# exit code 124 = killed by timeout after running fine (GOOD). Instant crash = problem.

In [ ]:
# Cell 5 - train the stock poca baseline, FRESH run (30M-step config)
# For the second session after a background-limit cutoff: add previous version's
# Output as an Input, copy its results/ into /kaggle/working/, and append --resume.
!mlagents-venv/bin/mlagents-learn repo/config/comm_mapoca/PushBlockCollabBaseline.yaml \
  --env=env/PBBaseline.x86_64 --no-graphics --num-envs=2 \
  --run-id=pushblock_poca_baseline_s1

In [ ]:
# Cell 6 - package results for download (runs after training stops)
!zip -r -q results_out.zip results/pushblock_poca_baseline_s1
!ls -la results_out.zip

In [ ]:
# Cell 7 - OPTIONAL: live TensorBoard peek (interactive sessions only)
%load_ext tensorboard
%tensorboard --logdir results